# Entrega 2 — Integração e Limpeza de Dados

**Projeto Final — Ciência de Dados (UTFPR)**

**Objetivo de pesquisa:** Predizer o output da produção agrícola nacional cruzando:
1. **Produção Agrícola (IBGE)** — Tabela 5457: área plantada por estado/cultura (2010–2024)
2. **Dados Climáticos (INMET)** — Médias mensais de ~450 estações meteorológicas
3. **Dados Econômicos (BCB)** — Série histórica do câmbio USD/BRL

---

In [90]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14

In [91]:
pasta_raw = Path('../data/weather/raw')
df_estacoes = pd.read_csv('../data/weather/dados_estacao.csv')
df_operante = df_estacoes[df_estacoes['situacao_status'] == 'Operante']

registros_clima = []

print(f"Processando {len(df_operante)} estações operantes...")

for idx, row in df_operante.iterrows():
    codigo = row['codigo_estacao']
    uf = row['estado_uf']
    
    arquivos_encontrados = list(pasta_raw.glob(f"dados_{codigo}_*.csv"))
    if not arquivos_encontrados:
        continue
        
    arquivo = arquivos_encontrados[0]
    try:
        df_est = pd.read_csv(
            arquivo, 
            skiprows=10, 
            sep=';', 
            decimal=',', 
            na_values=['null', '']
        )
        
        df_est = df_est.loc[:, ~df_est.columns.str.contains('^Unnamed')]
        df_est = df_est.dropna(how='all', axis=1)
        
        if df_est.empty or len(df_est.columns) < 7:
            continue
            
        df_est.columns = [
            'data_medicao', 'dias_precip', 'precip_total_mm',
            'pressao_media_mb', 'temp_media_c',
            'vento_max_ms', 'vento_media_ms'
        ]
        
        df_est['data_medicao'] = pd.to_datetime(df_est['data_medicao'], errors='coerce')
        df_est['ano'] = df_est['data_medicao'].dt.year
        df_est['mes'] = df_est['data_medicao'].dt.month
        df_est['uf'] = uf
        df_est['estacao'] = codigo
        
        num_cols = ['dias_precip', 'precip_total_mm', 'pressao_media_mb',
                    'temp_media_c', 'vento_max_ms', 'vento_media_ms']
        
        for col in num_cols:
            df_est[col] = pd.to_numeric(df_est[col], errors='coerce')
            
            if df_est[col].isna().all():
                df_est[col] = 0.0
            else:
                medias_mensais = df_est.groupby('mes')[col].transform('mean')
                if medias_mensais.isna().any():
                    medias_mensais = medias_mensais.fillna(df_est[col].mean())
                medias_mensais = medias_mensais.fillna(0.0)
                df_est[col] = df_est[col].fillna(medias_mensais)
                
        registros_clima.append(
            df_est[['uf', 'ano', 'mes', 'estacao'] + num_cols]
        )
    except Exception:
        pass

df_clima_bruto = pd.concat(registros_clima, ignore_index=True)
df_clima_bruto = df_clima_bruto[df_clima_bruto['ano'].between(2010, 2024)]


df_clima = df_clima_bruto.groupby(['uf', 'ano', 'mes']).agg(
    precip_total_mm  = ('precip_total_mm', 'mean'),
    dias_precip      = ('dias_precip', 'mean'),
    temp_media_c     = ('temp_media_c', 'mean'),
    pressao_media_mb = ('pressao_media_mb', 'mean'),
    vento_max_ms     = ('vento_max_ms', 'mean'),
    vento_media_ms   = ('vento_media_ms', 'mean'),
    n_estacoes       = ('estacao', 'nunique')
).reset_index()

media_hist = df_clima.groupby('uf')['precip_total_mm'].transform('mean')
std_hist   = df_clima.groupby('uf')['precip_total_mm'].transform('std')
df_clima['anomalia_precip'] = (df_clima['precip_total_mm'] - media_hist) / std_hist

output_clima_dir = Path('../data/weather/processed')
output_clima_dir.mkdir(parents=True, exist_ok=True)
df_clima.to_csv(output_clima_dir / 'medias_mensais_por_estado_ano.csv', index=False)

df_clima.head(10)

Processando 496 estações operantes...


,uf,ano,mes,precip_total_mm,dias_precip,temp_media_c,pressao_media_mb,vento_max_ms,vento_media_ms,n_estacoes,anomalia_precip
0,AC,2010,1,272.8,26.0,25.50,987.650000,7.00,1.60,2,0.994857
1,AC,2010,2,207.0,19.5,25.95,987.809091,6.20,1.40,2,0.379624
2,AC,2010,3,188.8,25.0,25.85,988.254545,7.55,1.25,2,0.209453
3,AC,2010,4,102.6,14.5,26.05,989.125000,5.95,1.30,2,-0.596522
4,AC,2010,5,36.6,15.0,24.45,990.209091,8.90,1.30,2,-1.213626
5,AC,2010,6,33.6,10.5,24.95,991.225000,5.55,1.15,2,-1.241676
6,AC,2010,7,20.8,9.0,23.45,991.540000,6.90,1.55,2,-1.361357
7,AC,2010,8,10.0,5.0,25.40,990.677778,6.75,1.45,2,-1.462337
8,AC,2010,9,55.9,8.0,26.85,988.668750,8.25,1.70,2,-1.033170
9,AC,2010,10,109.9,12.0,26.35,987.985000,7.70,1.70,2,-0.528267


In [92]:
UF_MAP = {
    'Rondônia': 'RO', 'Acre': 'AC', 'Amazonas': 'AM', 'Roraima': 'RR',
    'Pará': 'PA', 'Amapá': 'AP', 'Tocantins': 'TO', 'Maranhão': 'MA',
    'Piauí': 'PI', 'Ceará': 'CE', 'Rio Grande do Norte': 'RN',
    'Paraíba': 'PB', 'Pernambuco': 'PE', 'Alagoas': 'AL',
    'Sergipe': 'SE', 'Bahia': 'BA', 'Minas Gerais': 'MG',
    'Espírito Santo': 'ES', 'Rio de Janeiro': 'RJ', 'São Paulo': 'SP',
    'Paraná': 'PR', 'Santa Catarina': 'SC', 'Rio Grande do Sul': 'RS',
    'Mato Grosso do Sul': 'MS', 'Mato Grosso': 'MT', 'Goiás': 'GO',
    'Distrito Federal': 'DF'
}

CULTURAS = ['Cana_de_acucar', 'Mandioca', 'Milho', 'Soja']
ANOS = list(range(2010, 2025))

df_raw = pd.read_csv('../data/commodity/yearly-production-per-state-historical.csv',sep=';', header=None, skiprows=5)
        
if df_raw.iloc[:, -1].isna().all():
    df_raw = df_raw.iloc[:, :-1]

print(f"Shape bruto: {df_raw.shape}")

col_names = ['estado_nome']
for ano in ANOS:
    for cultura in CULTURAS:
        col_names.append(f'{cultura}_{ano}')
col_names = col_names[:len(df_raw.columns)]
df_raw.columns = col_names

records = []
for _, row in df_raw.iterrows():
    nome = str(row['estado_nome']).strip().strip('"')

    uf = UF_MAP.get(nome)
    if uf is None:
        for k, v in UF_MAP.items():
            if k.lower()[:5] == nome.lower()[:5]:
                uf = v
                break
    if uf is None:
        continue 

    for ano in ANOS:
        for cultura in CULTURAS:
            col = f'{cultura}_{ano}'
            if col not in row.index:
                continue

            val = row[col]
            if isinstance(val, str):
                val = val.strip().strip('"')
                if val in ['...', '', 'null']:
                    val = np.nan
                elif val == '-':
                    val = 0.0
                else:
                    try:
                        val = float(val.replace(',', '.'))
                    except ValueError:
                        val = np.nan
            elif pd.isna(val):
                val = np.nan
            else:
                val = float(val)

            records.append({
                'uf': uf, 'ano': ano, 'cultura': cultura,
                'area_plantada_ha': val
            })

df_agro = pd.DataFrame(records)

TIPO_CULTURA = {
    'Cana_de_acucar': 'Industrial', 'Mandioca': 'Raiz',
    'Milho': 'Grão', 'Soja': 'Grão'
}

REGIAO = {
    'AC': 'Norte', 'AM': 'Norte', 'AP': 'Norte', 'PA': 'Norte',
    'RO': 'Norte', 'RR': 'Norte', 'TO': 'Norte',
    'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste',
    'PB': 'Nordeste', 'PE': 'Nordeste', 'PI': 'Nordeste', 'RN': 'Nordeste',
    'SE': 'Nordeste',
    'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MS': 'Centro-Oeste',
    'MT': 'Centro-Oeste',
    'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
    'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul'
}

df_agro['tipo_cultura']  = df_agro['cultura'].map(TIPO_CULTURA)
df_agro['regiao']        = df_agro['uf'].map(REGIAO)

df_agro = df_agro.dropna(subset=['area_plantada_ha'])

df_agro.head(10)

Shape bruto: (288, 61)


,uf,ano,cultura,area_plantada_ha,tipo_cultura,regiao
0,RO,2010,Cana_de_acucar,3824.0,Industrial,Norte
1,RO,2010,Mandioca,29774.0,Raiz,Norte
2,RO,2010,Milho,169069.0,Grão,Norte
3,RO,2010,Soja,122743.0,Grão,Norte
4,RO,2011,Cana_de_acucar,3767.0,Industrial,Norte
5,RO,2011,Mandioca,30599.0,Raiz,Norte
6,RO,2011,Milho,160368.0,Grão,Norte
7,RO,2011,Soja,132300.0,Grão,Norte
8,RO,2012,Cana_de_acucar,3799.0,Industrial,Norte
9,RO,2012,Mandioca,27478.0,Raiz,Norte


In [81]:
df_cambio = pd.read_csv('../data/economic/dollar-historical-2000-2026.csv',
            sep=';', encoding='latin-1')


df_cambio.columns = ['data', 'cambio_venda']
df_cambio['data'] = pd.to_datetime(
    df_cambio['data'], format='%d/%m/%Y', errors='coerce'
)
df_cambio['cambio_venda'] = pd.to_numeric(
    df_cambio['cambio_venda'].astype(str)
    .str.replace(',', '.', regex=False),
    errors='coerce'
)

df_cambio['ano'] = df_cambio['data'].dt.year
df_cambio['mes'] = df_cambio['data'].dt.month

df_cambio = df_cambio[df_cambio['ano'].between(2010,2025)]

df_cambio['ano'] = df_cambio['ano'].astype(int)
df_cambio['mes'] = df_cambio['mes'].astype(int)

df_cambio = df_cambio.groupby(['ano', 'mes']).agg(
    cambio_medio  = ('cambio_venda', 'mean'),
    cambio_desvio = ('cambio_venda', 'std')
).reset_index().round(4)

In [85]:
df_cambio

,ano,mes,cambio_medio,cambio_desvio
0,2010,1,1.7798,0.0477
1,2010,2,1.8416,0.0241
2,2010,3,1.7858,0.0152
3,2010,4,1.7566,0.0124
4,2010,5,1.8132,0.0400
...,...,...,...,...
187,2025,8,5.4469,0.0421
188,2025,9,5.3674,0.0574
189,2025,10,5.3855,0.0477
190,2025,11,5.3409,0.0382


## INTEGRAÇÃO E PARQUET

In [93]:

df_meses = pd.DataFrame({'mes': range(1,13)})

print(len(df_agro))

df_agro = df_agro.merge(df_meses,how='cross')

print(len(df_agro))


12960
155520


In [94]:

df_integrado = df_agro.merge(df_clima, on=['uf', 'ano', 'mes'], how='left')

df_integrado = df_integrado.merge(df_cambio, on=['ano', 'mes'], how='left')


print(f"\nColunas ({len(df_integrado.columns)}):")
print(list(df_integrado.columns))
print(f"\nMissing values (%):")
missing = (df_integrado.isna().sum() / len(df_integrado) * 100).round(2)
print(missing[missing > 0])

df_integrado.head(10)


Colunas (17):
['uf', 'ano', 'cultura', 'area_plantada_ha', 'tipo_cultura', 'regiao', 'mes', 'precip_total_mm', 'dias_precip', 'temp_media_c', 'pressao_media_mb', 'vento_max_ms', 'vento_media_ms', 'n_estacoes', 'anomalia_precip', 'cambio_medio', 'cambio_desvio']

Missing values (%):
precip_total_mm     3.7
dias_precip         3.7
temp_media_c        3.7
pressao_media_mb    3.7
vento_max_ms        3.7
vento_media_ms      3.7
n_estacoes          3.7
anomalia_precip     3.7
dtype: float64


,uf,ano,cultura,area_plantada_ha,tipo_cultura,regiao,mes,precip_total_mm,dias_precip,temp_media_c,pressao_media_mb,vento_max_ms,vento_media_ms,n_estacoes,anomalia_precip,cambio_medio,cambio_desvio
0,RO,2010,Cana_de_acucar,3824.0,Industrial,Norte,1,389.921429,24.000000,24.75,966.850000,5.45,1.500000,2.0,1.671237,1.7798,0.0477
1,RO,2010,Cana_de_acucar,3824.0,Industrial,Norte,2,344.584615,12.000000,24.90,966.693333,6.10,1.480000,2.0,1.331756,1.8416,0.0241
2,RO,2010,Cana_de_acucar,3824.0,Industrial,Norte,3,326.407143,21.000000,25.50,966.900000,6.05,1.450000,2.0,1.195643,1.7858,0.0152
3,RO,2010,Cana_de_acucar,3824.0,Industrial,Norte,4,111.046154,7.500000,26.00,967.850000,5.15,1.550000,2.0,-0.416977,1.7566,0.0124
4,RO,2010,Cana_de_acucar,3824.0,Industrial,Norte,5,78.500000,6.000000,24.70,968.800000,5.80,1.700000,2.0,-0.660683,1.8132,0.0400
5,RO,2010,Cana_de_acucar,3824.0,Industrial,Norte,6,8.160000,0.500000,25.20,970.350000,5.90,1.500000,2.0,-1.187388,1.8065,0.0296
6,RO,2010,Cana_de_acucar,3824.0,Industrial,Norte,7,3.542857,1.642857,23.45,970.325000,5.50,1.595455,2.0,-1.221961,1.7696,0.0108
7,RO,2010,Cana_de_acucar,3824.0,Industrial,Norte,8,13.100000,2.076923,25.70,969.350000,5.85,1.577273,2.0,-1.150397,1.7596,0.0068
8,RO,2010,Cana_de_acucar,3824.0,Industrial,Norte,9,35.400000,4.500000,27.95,966.700000,7.80,1.600000,2.0,-0.983415,1.7187,0.0108
9,RO,2010,Cana_de_acucar,3824.0,Industrial,Norte,10,164.400000,16.500000,26.05,966.700000,7.75,1.650000,2.0,-0.017464,1.6835,0.0170


In [88]:
type(df_integrado)

pandas.core.frame.DataFrame

In [95]:
df_integrado.dropna()

output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)
df_integrado.to_csv(output_dir / 'dataset_integrado.csv', index=False)
df_integrado.to_parquet(output_dir / 'dataset_final.parquet', index=False)